In [16]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [17]:
X_train = pd.read_csv("../data/clean_X_train.csv")
X_test = pd.read_csv("../data/clean_X_test.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print(X_train.shape)
print(X_test.shape)

(5634, 22)
(1409, 22)


In [18]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=42
    ),

    "KNN": KNeighborsClassifier()
}

In [19]:
pipelines = {}

for name, model in models.items():

    pipelines[name] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

In [20]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTarget values:")
print(y_train.value_counts())

print("\nMissing values in X_train:")
print(X_train.isna().sum().sum())

print("\nData types:")
print(X_train.dtypes)

X_train: (5634, 22)
X_test: (1409, 22)
y_train: (5634,)
y_test: (1409,)

Target values:
Churn
0    4139
1    1495
Name: count, dtype: int64

Missing values in X_train:
0

Data types:
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
TenureGroup             str
NumServices           int64
AvgMonthlyCharge    float64
dtype: object


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipelines = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])
}

In [15]:
import joblib

preprocessor = joblib.load("../src/preprocessor.joblib")

print("Preprocessor loaded successfully")

Preprocessor loaded successfully


In [22]:
from sklearn.model_selection import cross_val_score

cv_results = {}

for name, pipe in pipelines.items():
    print("\n" + "=" * 60)
    print(f"Testing: {name}")
    print("=" * 60)

    try:
        scores = cross_val_score(
            pipe,
            X_train,
            y_train,
            cv=5,
            scoring="roc_auc",
            error_score="raise"
        )

        cv_results[name] = scores

        print("AUC scores:", scores)
        print("Mean CV AUC:", scores.mean())
        print("Std CV AUC:", scores.std())

    except Exception as e:
        print(f"ERROR in {name}")
        print(type(e)._name_)
        print(str(e))


Testing: Logistic Regression
AUC scores: [0.86668929 0.85752831 0.85251967 0.83903672 0.82454413]
Mean CV AUC: 0.8480636248800953
Std CV AUC: 0.014771619162766435


In [23]:
results = []

reports = {}

for name, pipe in pipelines.items():

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    y_prob = pipe.predict_proba(X_test)[:,1]

    reports[name] = classification_report(
        y_test,
        y_pred
    )

    results.append({

        "Model": name,

        "Accuracy":
        accuracy_score(y_test, y_pred),

        "Precision":
        precision_score(y_test, y_pred),

        "Recall":
        recall_score(y_test, y_pred),

        "F1":
        f1_score(y_test, y_pred),

        "ROC_AUC":
        roc_auc_score(y_test, y_prob)
    })

In [24]:
metrics_df = pd.DataFrame(results)

metrics_df = metrics_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

metrics_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.798439,0.65625,0.505348,0.570997,0.844966


In [25]:
for model, report in reports.items():

    print("="*50)
    print(model)
    print("="*50)

    print(report)

Logistic Regression
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1035
           1       0.66      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [26]:
winner = metrics_df.iloc[0]["Model"]

print("Best Model:", winner)

Best Model: Logistic Regression
